In [ ]:
import openmeteo_requests
import openmeteo_requests
import pandas as pd
import numpy as np
import requests_cache
from retry_requests import retry
from pathlib import Path
import time


INPUT_FILE  = "/kaggle/input/datasets/kareemabosamra/uk-road-casualty/dft-road-casualty-statistics-collision-provisional-2025.csv"
OUTPUT_DIR  = "/kaggle/working/weather_batches/"
FINAL_OUTPUT = "/kaggle/working/road_casualties_with_weather.csv"  

START_FROM_GROUP = 1800 


WEATHER_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "snow_depth",
    "weather_code",
    "wind_speed_10m",
    "wind_direction_10m",
    "wind_gusts_10m",
    "visibility",
    "surface_pressure",
    "cloud_cover",
    "is_day",
]

API_URL = "https://archive-api.open-meteo.com/v1/archive"
LAT_LON_PRECISION = 1
SLEEP_BETWEEN_CALLS = 2

MAX_RETRIES = 5
RETRY_WAIT_SECONDS = 60

ENABLE_CHECKPOINTS = True
CHECKPOINT_EVERY   = 50  # Save batch every 50 groups
CHECKPOINT_FILE    = "/kaggle/working/weather_checkpoint_state.csv"
PROGRESS_FILE      = "/kaggle/working/weather_progress.txt"


Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

cache_session = requests_cache.CachedSession(".weather_cache", expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.5)
openmeteo     = openmeteo_requests.Client(session=retry_session)


def load_checkpoint():
    """Return last completed group index from disk, or -1."""
    if not ENABLE_CHECKPOINTS:
        return -1
    pr_path = Path(PROGRESS_FILE)
    if pr_path.exists():
        last_idx = int(pr_path.read_text().strip())
        print(f" last completed group: {last_idx:,}")
        return last_idx
    return -1


def save_progress(last_idx: int):
    """Update the progress marker."""
    if not ENABLE_CHECKPOINTS:
        return
    Path(PROGRESS_FILE).write_text(str(last_idx))


def save_batch(weather_rows: list, start_idx: int, end_idx: int):
    """
    Save accumulated weather data to a batch file.
    Filename format: weather_batch_{start_idx:05d}_{end_idx:05d}.csv
    """
    if not weather_rows:
        print(f"No data to save for batch {start_idx}-{end_idx}")
        return
    
    combined = pd.concat(weather_rows, ignore_index=True)
    batch_file = Path(OUTPUT_DIR) / f"weather_batch_{start_idx:05d}_{end_idx:05d}.csv"
    combined.to_csv(batch_file, index=False)
    print(f" Saved batch -> {batch_file.name}  ({len(combined):,} rows)")


def get_last_batch_end():
    """
    Scan OUTPUT_DIR for existing batch files and return the highest end index.
    This allows resuming even if checkpoint progress file is missing.
    """
    batch_files = list(Path(OUTPUT_DIR).glob("weather_batch_*.csv"))
    if not batch_files:
        return -1
    
    max_end = -1
    for bf in batch_files:
        try:
            parts = bf.stem.split("_")
            end_idx = int(parts[-1])
            max_end = max(max_end, end_idx)
        except:
            continue
    
    if max_end >= 0:
        print(f"Found existing batches — last batch ends at group: {max_end:,}")
    return max_end


def clear_checkpoint():
    """Remove checkpoint files after successful completion."""
    for p in [CHECKPOINT_FILE, PROGRESS_FILE]:
        path = Path(p)
        if path.exists():
            path.unlink()
    print("   Checkpoint files cleared.")



print("=" * 70)
print("Loading dataset ...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"  Rows: {len(df):,}  |  Columns: {len(df.columns)}")


df["datetime"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str),
    dayfirst=True,
    errors="coerce",
)
df["accident_hour"] = df["datetime"].dt.floor("h")
df["_year"]         = df["datetime"].dt.year

df["_lat_r"] = df["latitude"].round(LAT_LON_PRECISION)
df["_lon_r"] = df["longitude"].round(LAT_LON_PRECISION)


groups = (
    df[["_lat_r", "_lon_r", "_year"]]
    .drop_duplicates()
    .dropna()
    .reset_index(drop=True)
)

old_call_count = (
    df[["_lat_r", "_lon_r"]].assign(_date=df["datetime"].dt.date)
    [["_lat_r", "_lon_r", "_date"]].drop_duplicates()
    .shape[0]
)

print(f"\n  API call reduction summary:")
print(f"    Old strategy  (per location × date) : {old_call_count:,} calls")
print(f"    New strategy  (per location × year) : {len(groups):,} calls")
print(f"    Reduction     : {old_call_count / max(len(groups), 1):.1f}x fewer calls\n")


def fetch_year(lat: float, lon: float, year: int):
    """Fetch a full year of hourly weather for one location."""
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": f"{year}-01-01",
        "end_date":   f"{year}-12-31",
        "hourly":     WEATHER_VARS,
        "timezone":   "UTC",
    }
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            responses = openmeteo.weather_api(API_URL, params=params)
            resp      = responses[0]
            hourly    = resp.Hourly()

            times = pd.date_range(
                start     = pd.to_datetime(hourly.Time(),    unit="s", utc=True),
                end       = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
                freq      = pd.Timedelta(seconds=hourly.Interval()),
                inclusive = "left",
            )

            records = {"datetime_utc": times}
            for i, var in enumerate(WEATHER_VARS):
                records[f"wx_{var}"] = hourly.Variables(i).ValuesAsNumpy()

            wx = pd.DataFrame(records)
            wx["_lat_r"] = lat
            wx["_lon_r"] = lon
            wx["accident_hour"] = wx["datetime_utc"].dt.floor("h")

            if attempt > 1:
                print(f"    ✓  Success on attempt {attempt}")
            return wx

        except Exception as e:
            if attempt < MAX_RETRIES:
                print(f" Attempt {attempt}/{MAX_RETRIES} failed for (lat={lat}, lon={lon}, year={year})")
                print(f"Error: {e}")
                print(f"  Waiting {RETRY_WAIT_SECONDS}s before retry...")
                time.sleep(RETRY_WAIT_SECONDS)
            else:
                print(f"All {MAX_RETRIES} attempts failed for (lat={lat}, lon={lon}, year={year})")
                print(f"Final error: {e}")
                return None
    return None


print("=" * 70)
print("Determining start group ...")

if START_FROM_GROUP is not None:
    start_from = START_FROM_GROUP
    print(f" Manual start requested: START_FROM_GROUP = {start_from}")
    if start_from > 0:
        print(f" Make sure batches 0-{start_from-1} exist if you plan to merge later!")
else:
    checkpoint_last = load_checkpoint()
    batch_last = get_last_batch_end()
    
    start_from = max(checkpoint_last, batch_last) + 1
    
    if start_from > 0:
        print(f" Auto-resuming from group: {start_from}")
    else:
        print(f"Starting from group: 0")

if start_from >= len(groups):
    print(f"\n All groups already processed! (start_from={start_from} >= total={len(groups)})")
    print(f"To re-run, set START_FROM_GROUP = 0 or delete batch files.")
    exit(0)

print(f"  Total groups: {len(groups):,}  |  Remaining: {len(groups) - start_from:,}")
print("=" * 70)


batch_buffer = []
batch_start_idx = start_from

remaining = groups.iloc[start_from:]
total = len(groups)

for offset, (_, row) in enumerate(remaining.iterrows()):
    idx = start_from + offset
    lat, lon, year = row["_lat_r"], row["_lon_r"], int(row["_year"])

    if offset % 10 == 0 or offset == 0:
        pct = idx / total * 100
        print(f"\n  [{pct:5.1f}%]  Group {idx:,} / {total:,}  "
              f"→  lat={lat}, lon={lon}, year={year}")

    wx = fetch_year(lat, lon, year)

    if wx is not None:
        batch_buffer.append(wx)
    else:
        print(f"  ⚠  Skipping group {idx} due to fetch failure")

    groups_in_batch = idx - batch_start_idx + 1
    
    if groups_in_batch >= CHECKPOINT_EVERY or idx == total - 1:
        batch_end_idx = idx
        print(f"\n  📦  Batch complete: groups {batch_start_idx}-{batch_end_idx}")
        save_batch(batch_buffer, batch_start_idx, batch_end_idx)
        save_progress(batch_end_idx)
        
        batch_buffer = []
        batch_start_idx = batch_end_idx + 1

    time.sleep(SLEEP_BETWEEN_CALLS)

print("\n" + "=" * 70)
print("  ✅  Done fetching all groups!")
print("=" * 70)


def merge_all_batches():
    """Load all batch files and merge with original dataset."""
    print("\n" + "=" * 70)
    print("Merging all batches into final dataset ...")
    
    batch_files = sorted(Path(OUTPUT_DIR).glob("weather_batch_*.csv"))
    
    if not batch_files:
        print("  ⚠  No batch files found!")
        return
    
    print(f"  Found {len(batch_files)} batch file(s)")
    
    all_weather = []
    for bf in batch_files:
        print(f"    Loading {bf.name} ...")
        batch_df = pd.read_csv(bf, parse_dates=["datetime_utc"])
        all_weather.append(batch_df)
    
    weather_df = pd.concat(all_weather, ignore_index=True)
    print(f"  Total weather rows: {len(weather_df):,}")
    
    if weather_df["accident_hour"].dt.tz is None:
        weather_df["accident_hour"] = weather_df["accident_hour"].dt.tz_localize("UTC")
    
    wx_cols = [f"wx_{v}" for v in WEATHER_VARS]
    weather_lookup = weather_df[["_lat_r", "_lon_r", "accident_hour", *wx_cols]].copy()
    
    df_merge = df.copy()
    df_merge["accident_hour"] = df_merge["accident_hour"].dt.tz_localize("UTC")
    
    print("  Merging weather into dataset ...")
    df_out = df_merge.merge(
        weather_lookup,
        on=["_lat_r", "_lon_r", "accident_hour"],
        how="left",
    )
    
    df_out.drop(columns=["_lat_r", "_lon_r", "_year", "accident_hour"], inplace=True)
    
    df_out.to_csv(FINAL_OUTPUT, index=False)
    print(f"\n  💾  Saved final dataset -> {FINAL_OUTPUT}")
    print(f"      Shape: {df_out.shape}")
    
    filled = df_out[wx_cols[0]].notna().sum()
    print(f"      Rows with weather: {filled:,} / {len(df_out):,} ({filled/len(df_out)*100:.1f}%)")
    
    if ENABLE_CHECKPOINTS:
        clear_checkpoint()

merge_all_batches()

print("\n" + "=" * 70)
print("Batch files saved in:", OUTPUT_DIR)
print("To resume later, set START_FROM_GROUP or delete progress file.")
print("=" * 70)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.6/208.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.9/719.9 kB 18.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.4/399.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.5 MB/s eta 0:00:00:00:01
✅ Packages installed successfully
Loading dataset ...
  Rows: 48,472  |  Columns: 44

  API call reduction summary:
    Old strategy  (per location × date) : 34,989 calls
    New strategy  (per location × year) : 2,341 calls
    Reduction     : 14.9x fewer calls

Determining start group ...
  🎯  Manual start requested: START_FROM_GROUP = 1800
      ⚠  Make sure batches 0-1799 exist if you plan to merge later!
  Total groups: 2,341

KeyboardInterrupt: 